<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">

<div class="alert alert-block alert-warning">
<strong>Notice Regarding Use and Access</strong><br>
Training materials, including Jupyter notebooks, are provided for educational use under the <a href="https://nam11.safelinks.protection.outlook.com/?url=https%3A%2F%2Fwww.nvidia.com%2Fen-us%2Fagreements%2Fenterprise-services%2Fnvidia-training-services-terms-and-conditions%2F&data=05%7C02%7Cdsheahen%40nvidia.com%7C195db190675a4c9b3ebd08ddb804cb3e%7C43083d15727340c1b7db39efd9ccc17a%7C0%7C0%7C638869052943842297%7CUnknown%7CTWFpbGZsb3d8eyJFbXB0eU1hcGkiOnRydWUsIlYiOiIwLjAuMDAwMCIsIlAiOiJXaW4zMiIsIkFOIjoiTWFpbCIsIldUIjoyfQ%3D%3D%7C0%7C%7C%7C&sdata=Smt58g4tBoUXLTwTaOrs3iy3ylb7llHMii0M1Pvaiik%3D&reserved=0">NVIDIA Training Services Terms and Conditions</a>.
The Docker containers supporting these notebooks are restricted to the NVIDIA learning environment. Downloading, extracting, or redistributing it is strictly prohibited. By using this platform, you agree not to access the container. Violations may result in access suspension and legal action.
</div>

# MONAI Label Radiology App - Slicer Use Case Example
***The Active Learning Process with MONAI Label***

In this notebook, we provide a hello world example of the MONAI Label use case. Using Spleen segmentation in the Radiology app as the demonstration and 3D Slicer as the client viewer, we show how the MONAI Label workflow serves as an interacitve AI-Assisted tool for labeling CT scans. 

<img src="./images/monailabel_arch_slicer.png" width="1000"/>

<p>
    <img src="../toolkit/tutorials/monailabel/figures/monailabel_radiology_3dslicer/teaser_img.png" width="1000"/>
</p>

---

> **Note**: This notebook is an adaptation of the [monailabel_radiology_3dslicer.ipynb](../toolkit/tutorials/monai/monailabel/monailabel_HelloWorld_radiology_3dslicer.ipynb) 
notebook released with the MONAI Toolkit. 
We have removed cells for setup and data download which are already loaded for you in this lab. 

---

### Table of Contents
**[0.0 Slicer Setup](#0.0-Slicer-Setup)** <br>
**[1.0 noVNC with Preinstalled Program](#1.0-noVNC-with-Preinstalled-Program)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[1.1 Open noVNC](#1.1-Open-noVNC) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[1.2 Start MONAI Label Server](#1.2-Start-MONAI-Label-Server) <br>
**[2.0 Auto-Segmentation with 3D Slicer](#2.0-Auto-Segmentation-with-3D-Slicer)** <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.1 Open Start 3D Slicer and MONAI Label Plugin](#2.1-Open-Start-3D-Slicer-and-MONAI-Label-Plugin) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.2 Start 3D Slicer and MONAI Label Plugin](#2.2-Start-3D-Slicer-and-MONAI-Label-Plugin) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.3 Select Model and Fetch Data](#2.3-Select-Model-and-Fetch-Data) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.4 Run Auto-Segmentation](#2.4-Run-Auto-Segmentation) <br>
&nbsp;&nbsp;&nbsp;&nbsp;[2.5 Edit Annotations and Submit Label](#2.5-Edit-Annotations-and-Submit-Label) <br>
**[3.0 Active Learning with 3D Slicer](#3.0-Active-Learning-with-3D-Slicer)** <br>
**[4.0 Repeat the Interactive Labeling Process](#4.0-Repeat-the-Interactive-Labeling-Process)** <br>
**[5.0 Slicer Conclusion](#5.0-Slicer-Conclusion)** <br>
**[6.0 Slicer Next Steps](#6.0-Slicer-Next-Steps)** <br>

---
# 0.0 Slicer Setup  
Begin by importing some libraries used in this notebook. 

In [ ]:
import ipywidgets
import threading
import os
HOST_NAME=os.getenv('HOST_NAME')
HOST_IP=os.getenv('HOST_IP')
DLI_DATA_DIR=os.getenv('DLI_DATA_DIR')
os.chdir(DLI_DATA_DIR)


if True or len(HOST_NAME)==0:
    from requests import get
    HOST_NAME=get('https://api.ipify.org').text
    HOST_IP=HOST_NAME
print(f" HOST_NAME is {HOST_NAME}  with IP {HOST_IP} ")

In [ ]:
HOST_NAME

---
# 1.0 noVNC with Preinstalled Program 
This DLI lab already has a noVNC instance running with preinstalled programs:
- 3d Slicer
- QuPath
- ImageJ

## 1.1 Open noVNC
Run the cell below; then click the link.

<div class="alert alert-block alert-info">
<b>Password:</b>gtc25
</div>    

In [ ]:
url_text='<h1><center><a href="http://'+HOST_NAME+'/x11/" target="_blank">'
url_text=url_text+'<img src="https://user-images.githubusercontent.com/15837524/103544638-6999fa80-4e6e-11eb-94c4-5c42cd2f63f8.png" width="200" height="100"/>'
url_text=url_text+'<br>Click to open 3D Slicer <br> in a new window</a></center></h1>'
ipywidgets.HTML(url_text)

## 1.2 Start MONAI Label Server

In [ ]:
!monailabel start_server --app apps/radiology --studies datasets/Task09_Spleen/imagesTr --conf models segmentation_spleen
# !monailabel start_server --app apps/radiology --studies datasets/Task09_Spleen/imagesTr --conf models segmentation
# !monailabel start_server --app apps/radiology --studies datasets/Task03_Liver --conf models deepgrow_3d,deepgrow_2d
# !monailabel start_server --app apps/monaibundle --studies datasets/Task09_Spleen/imagesTr --conf models vista3d

---
# 2.0 Auto-Segmentation with 3D Slicer

Below is a step-by-step tutorial on using 3D Slicer for auto spleen segmentation. 
A pretrained model is provided and loaded by MONAI Label server.


## 2.1 Open Start 3D Slicer and MONAI Label Plugin
In the VNC instance you should have Slicer automatically opened. 
From there in the module bar --> Active learning --> MONAI Label as show below 

<img src="./images/slicer_open_plugin.png" width="600"/>
 

## 2.2 Start 3D Slicer and MONAI Label Plugin

Type the MONAI server IP:Port in the MONAI Label server. 
Since we are using Docker Compose for this lab, the VNC instance can resolve the IP using an internal name.
<div class="alert alert-danger"> 
MONAI Label server should be: 'http://dli-lab:8000
</div>

Connect and check: MONAI Label Server Address/Name, radiology app, and confirm 3D Slicer is connected to MONAI Label server correctly. If not, fill in the correct server address and port.

<img src="../toolkit/tutorials/monailabel/figures/monailabel_radiology_3dslicer/slicer1.png" alt="slicer1" width="1000">


## 2.3 Select Model and Fetch Data

- Select the model users would like to use, such as "**segmentation_spleen**" in this tutorial, click **next sample** to load image data from study datastore.

<img src="../toolkit/tutorials/monailabel/figures/monailabel_radiology_3dslicer/slicer2.png" alt="slicer2" width="1000">


## 2.4 Run Auto-Segmentation

- MONAI Label has provided pretrained model weights for the spleen segmentation model. Click **run** to do inference on the current loaded data. 

- Note: the pretrained models will be downloaded when starting MONAI Label server in the "**model**" folder. For instance, in this use case, the downloaded pretrained model is saved at "**apps/radiology/model/pretrained_segmentation_spleen.pt**".

<img src="../toolkit/tutorials/monailabel/figures/monailabel_radiology_3dslicer/slicer3.png" alt="slicer3" width="1000">


## 2.5 Edit Annotations and Submit Label

- Users can edit the inference labels manually using Segment Editor, the tool for adjusting annotation. Once done with the annotation, click **Submit Label** to save the ground truth label to the file system.

- Note: the final annotations will be saved to the "**labels/final**" folder in the study dataset.  For example, in this use case, the ground truth label will be saved to "**datasets/Task09_Spleen/imagesTr/labels/final**"


#### AI assisted corrections:

<img src="../toolkit/tutorials/monailabel/figures/monailabel_radiology_3dslicer/scribbles.png" alt="scribbles" width="1000">


#### Corrections with Segment Editor:

<img src="../toolkit/tutorials/monailabel/figures/monailabel_radiology_3dslicer/slicer4.png" alt="slicer4" width="1000">


---
# 3.0 Active Learning with 3D Slicer

Active learning and interactive fine-tuning models have highlighted features in MONAI Label. Users can train their models anytime when new annotated labels are saved. When you click the **Train** button, the MONAI Label server will fetch the saved final ground truth label and fine-tune the prior model. 

<img src="../toolkit/tutorials/monailabel/figures/monailabel_radiology_3dslicer/slicer5.png" alt="slicer5" width="1000">

Users can do auto-segmentation using the latest fine-tuned model. 
The active learning process will select unlabeld batch data.

When next image is selected, the already trained images will be marked as labeled data, and won't be selected for the next labeling batch. Active learning strategies such as "first/random" will be used for selecting which unlabeled data to fetch. 

For the general active learning workflow design, see the slide: 

<img src="../toolkit/tutorials/monailabel/figures/monailabel_radiology_3dslicer/activeLearning.png" alt="activeL" width="1000">


---
# 4.0 Repeat the Interactive Labeling Process 

Repeat the fetch data and active learning process until all unlabeled data are annotated and trained. 

For each training loop, the new best metric model will be saved in "**model/<model_name>.pt**". In this use case, ""**apps/radiology/model/segmentation_spleen.pt**" is saved. For detailed training stats and step models, refer to "**apps/radiology/model/segmentation_spleen/train_01**", 

---
# 5.0 Slicer Conclusion

In this lab, you learned:
- Basic usage of MONAI Label with the radiology app, and a spleen segmentation task was introduced
- The active learning process, auto-segmentation, segmentation editor, submit labels, and save models.
- The interactive learning process when labeling a dataset can achieve incremental improvements and enable improved understanding of medical data for researchers and clinicians. 


---
# 6.0 Slicer Next Steps
1. This DLI lab has 3D Slicer set up inside a vnc for convenience. 
You can install 3D Slicer locally on your machine and connect from https://download.slicer.org/.
1. Next, you need to install the MONAI Label plugin. 
For step-by-step instructions, see https://docs.monai.io/projects/label/en/latest/quickstart.html   
1. You could try using other models from the MONAI Zoo. For example, the whole body CT segmentation using the code below:

In [ ]:
!monailabel start_server \
    --app apps/monaibundle \
    --studies datasets/Task09_Spleen/imagesTr \
    --conf models wholeBody_ct_segmentation \
    --conf zoo_source github \
    --conf zoo_repo Project-MONAI/model-zoo/hosting_storage_v1 

<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">